# 1. Prereq Setup

In [1]:
! pip install --upgrade langchain langchain-core langchain-community langchain-ollama dspy pandas openpyxl tqdm --quiet

1. We need Ollama to be installed on the system for serving llms to run this script. I can be downloaded from https://ollama.com/download
2. Once downloaded, run the following command to start the local server with the desired model (here we use gemma3:12b-it-qat and gemma3:4b):/ ollama serve gemma3:12b-it-qat --port 11434 and ollama serve gemma3:4b --port 11435
##### Note: The whole code can be put in python, shell scripting and README files to make it production ready but for the purpose of this assignment we will be using notebooks since they are easier to walk through in furhter discussions.
##### Input data parquet files have not been committed due to large volume. Please download from https://github.com/amazon-science/esci-data  

# 2. Data Prepration

In [2]:
import pandas as pd

In [3]:
# loading the datasets
df_examples = pd.read_parquet('./data/shopping_queries_dataset_examples.parquet')
df_products = pd.read_parquet('./data/shopping_queries_dataset_products.parquet')
df_sources = pd.read_csv("./data/shopping_queries_dataset_sources.csv")

In [4]:
print(df_examples.shape)
df_examples.head()

(2621288, 9)


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train


In [5]:
print(df_products.shape)
df_products.head()

(1814924, 7)


,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,B079VKKJN7,"11 Degrees de los Hombres Playera con Logo, Ne...",Esta playera con el logo de la marca Carrier d...,11 Degrees Negro Playera con logo\nA estrenar ...,11 Degrees,Negro,es
1,B079Y9VRKS,Camiseta Eleven Degrees Core TS White (M),None,None,11 Degrees,Blanco,es
2,B07DP4LM9H,11 Degrees de los Hombres Core Pull Over Hoodi...,La sudadera con capucha Core Pull Over de 11 G...,11 Degrees Azul Core Pull Over Hoodie\nA estre...,11 Degrees,Azul,es
3,B07G37B9HP,11 Degrees Poli Panel Track Pant XL Black,None,None,11 Degrees,None,es
4,B07LCTGDHY,11 Degrees Gorra Trucker Negro OSFA (Talla úni...,None,None,11 Degrees,Negro (,es


In [6]:
print(df_sources.shape)
df_sources.head()

(130652, 2)


,query_id,source
0,0,other
1,1,negations
2,2,negations
3,3,negations
4,4,behavioral


In [7]:
# preparing the merged dataframe to be used for the tasks. 
df_examples_products = pd.merge(
    df_examples,
    df_products,
    how='left',
    left_on=['product_locale','product_id'],
    right_on=['product_locale', 'product_id']
)

In [8]:
# data filtering to only work with given queries. 
df_data = df_examples_products[(df_examples_products['query'].isin(["aa batteries 100 pack", "kodak photo paper 8.5 x 11 glossy", "dewalt 8v max cordless screwdriver kit, gyroscopic"])) &  
                     (df_examples_products['esci_label'].astype(str)=='E')] # Taking only required data

In [9]:
print(df_data.shape)
df_data.head()

(24, 14)


,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color
142651,142651,aa batteries 100 pack,6014,B01G1RYHAO,us,E,0,1,train,Energizer Advanced AA Alkaline Bulk Battery - ...,Bulk Packaging,Bulk Packaging,Energizer,White/Brown
142652,142652,aa batteries 100 pack,6014,B07FP5DNBG,us,E,0,1,train,"IMPECCA AA Batteries, All Purpose Alkaline Bat...",<p>The AA alkaline battery is one of the most ...,Packaging may VARY! AA 1.5 volt alkaline batte...,Impecca,PLATINUM
142653,142653,aa batteries 100 pack,6014,B07F7RH8D4,us,E,0,1,train,Allmax AA Maximum Power Alkaline Batteries (10...,None,★ Maximum Power – Allmax Maximum Power AA Batt...,Allmax Battery,None
142659,142659,aa batteries 100 pack,6014,B01B8R6PF2,us,E,0,1,train,Amazon Basics 100 Pack AA High-Performance Alk...,None,IN THE BOX: 100-pack of 1.5 volt AA alkaline b...,Amazon Basics,None
142660,142660,aa batteries 100 pack,6014,B00LHSAARW,us,E,0,1,train,"Rayovac AA Alkaline Double A Batteries, 60 Count",None,60 pack of Rayovac High Energy Alkaline AA Bat...,Rayovac,None


In [10]:
df_data['query'].value_counts() # checking query balance. 

query
kodak photo paper 8.5 x 11 glossy                     10
aa batteries 100 pack                                  8
dewalt 8v max cordless screwdriver kit, gyroscopic     6
Name: count, dtype: int64

In [11]:
# data Preprocessing. Lets extract tags realted to product that will be useful for LLM to perform tasks. 
import dspy 
from tqdm import tqdm

lm = dspy.LM("ollama_chat/gemma3:12b-it-qat", api_base="http://localhost:11434", api_key="") # both of the models used in my case are hosted on 11434 port but if you server on different port please change accordingly
dspy.configure(lm=lm, temperature=0)

class SearchTagsGen(dspy.Signature):
    """You are an e-commerce tags extractor. Your task is to extract relevant tags from product description which can be usefull to add filtering. These tags are also usefull in defining product specifications. 

        Rules:
        - keep the output tags between 1-2 word tags each.
        - add special attention to product specifications mentioned in the product title, description, and bullet points while extracting tags
        - avoid unnecessary elaboration or verbosity.
        - Never hallucinate specs or infer unstated details.

        Output format (JSON):
          "tags": list of string

        Examples:

        1. Inaccurate:
        Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
        Output:
        "tags": ["King size", "Quilted", "Pillow shams included", "Moose Lodge design"]

        2. Contradiction:
        Product: DEWALT 8V MAX, Gyroscopic
        Output:
        "tags": ["8V MAX", "Cordless", "Screwdriver kit", "Gyroscopic"]

      Use the above examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.
      """

    product_title: str=  dspy.InputField(desc = "Title of the product associated with the query")
    product_description: str = dspy.InputField(desc = "Description of the product associated with the query")
    product_bullet_points: str = dspy.InputField(desc = "Bullet points of the product associated with the query")
    tags: list[str] = dspy.OutputField(desc = "product taags extracted from the product information in list format.")

module_tags = dspy.Predict(SearchTagsGen)


# populating the data for all examples
df_data['tags'] = None
for i in tqdm(range(len(df_data))):
    query= df_data.iloc[i]['query']
    product_title = df_data.iloc[i]['product_title']
    product_description = df_data.iloc[i]['product_description']
    product_bullet_points = df_data.iloc[i]['product_bullet_point']

    response_tags = module_tags(
        query=query,
        product_title=product_title,
        product_description=product_description,
        product_bullet_points=product_bullet_points
    )

    df_data.at[df_data.index[i], 'tags'] = response_tags.tags

/var/folders/8_/qq_4gly11h51l7vw7rmfl8980000gn/T/ipykernel_11986/2448782943.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_data['tags'] = None
100%|██████████| 24/24 [00:01<00:00, 23.79it/s]


In [12]:
df_data.head()

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color,tags
142651,142651,aa batteries 100 pack,6014,B01G1RYHAO,us,E,0,1,train,Energizer Advanced AA Alkaline Bulk Battery - ...,Bulk Packaging,Bulk Packaging,Energizer,White/Brown,"[AA battery, Alkaline, Bulk, 100 count]"
142652,142652,aa batteries 100 pack,6014,B07FP5DNBG,us,E,0,1,train,"IMPECCA AA Batteries, All Purpose Alkaline Bat...",<p>The AA alkaline battery is one of the most ...,Packaging may VARY! AA 1.5 volt alkaline batte...,Impecca,PLATINUM,"[AA batteries, Alkaline batteries, 1.5V, 100-p..."
142653,142653,aa batteries 100 pack,6014,B07F7RH8D4,us,E,0,1,train,Allmax AA Maximum Power Alkaline Batteries (10...,None,★ Maximum Power – Allmax Maximum Power AA Batt...,Allmax Battery,None,"[AA Battery, Alkaline, 1.5V, Long-lasting, Lea..."
142659,142659,aa batteries 100 pack,6014,B01B8R6PF2,us,E,0,1,train,Amazon Basics 100 Pack AA High-Performance Alk...,None,IN THE BOX: 100-pack of 1.5 volt AA alkaline b...,Amazon Basics,None,"[AA batteries, Alkaline, 1.5 volt, 10-year, Le..."
142660,142660,aa batteries 100 pack,6014,B00LHSAARW,us,E,0,1,train,"Rayovac AA Alkaline Double A Batteries, 60 Count",None,60 pack of Rayovac High Energy Alkaline AA Bat...,Rayovac,None,"[AA batteries, Alkaline, 60 count, Rayovac, Lo..."


In [13]:
df_data.to_excel("./processed_data/shopping_queries_dataset_esci_E_with_tags.xlsx", index=False)

### Data Conclusions

1. We are using only 3 queries as mentioned in the task description further. 
2. A total of 24 query-product are used for this tasks. 
3. Altough the data is huge but were are only using small subset of it beacuse it would be near impossible to run such large data even with tinly LLMs (~1b-12b) on a 16GB Macbook Pro
4. The data contains the User query, product title, product description and product bullet points which would be useful inputs for out AI solution. 

# 3. Building Search Auditor Using single LLM 

In [14]:
df_data = pd.read_excel("./processed_data/shopping_queries_dataset_esci_E_with_tags.xlsx")

### 3.1 DSPY Solution 

In [15]:
import dspy
from tqdm import tqdm

In [16]:
lm = dspy.LM("ollama_chat/gemma3:12b-it-qat", api_base="http://localhost:11434", api_key="")
dspy.configure(lm=lm, temperature=0)

In [17]:
# defining the tasks for LLM with appropriate prompts 
class SearchAuditor(dspy.Signature):
    """You are an e-commerce relevance auditor using the ESCI labeling framework. Check if the given query–product pair labeled “E” (Exact match) truly matches the definition:

        Rules:
        - “E” (Exact) = product title, description, and bullet points fully satisfy query intent. Extra or missing info is okay if not contradictory.
        - NOT “E” = product contradicts query (e.g., includes excluded items, wrong specs, wrong dimensions/colors/variants).
        - Ambiguous info = if product doesn’t contradict query, keep as “E”.
        - Never hallucinate specs or infer unstated details.
        - When “E” is incorrect, rewrite the query to accurately match the product title, product description and bullet points.
        - add special attention to product specifications mentioned in the product title, description, and bullet points while reforming queries

        Output format (JSON):
          "is_accurate": True/False,
          "reformulated_query": string or null

        Examples:

        1. Inaccurate:
        Query: 105" x 115" king spread without pillow shams
        Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
        Output:
        "is_accurate": false, "reformulated_query": "105\" x 115\" king spread with pillow shams"

        2. Accurate (extra info allowed):
        Query: vitamina c
        Product: Vitamina C Liposomal 400 mg, 90 capsules, extra details
        Output:
        "is_accurate": true, "reformulated_query": null

        3. Accurate (missing info allowed):
        Query: 12 month clothes girls
        Product: Newborn kids romper + headband, no gender info
        Output:
        "is_accurate": true, "reformulated_query": null

        4. Contradiction:
        Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
        Product: DEWALT 8V MAX, Gyroscopic
        Output:
        "is_accurate": false, "reformulated_query": "dewalt 8v max cordless screwdriver kit, gyroscopic"

      Use the above examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.
      """

    query: str = dspy.InputField(desc = "User search query entered on platform")
    product_title: str=  dspy.InputField(desc = "Title of the product associated with the query")
    product_description: str = dspy.InputField(desc = "Description of the product associated with the query")
    product_bullet_points: str = dspy.InputField(desc = "Bullet points of the product associated with the query")
    esci_label: str= dspy.InputField(desc = "ESCI label assigned to the query-product pair, always 'E' in this case")
    product_brand: str= dspy.InputField(desc = "Brand of the product associated with the query")
    product_color: str= dspy.InputField(desc = "Color of the product associated with the query")
    tags: list[str]= dspy.InputField(desc = "List of product tags extracted from the product information")
    is_accurate: bool = dspy.OutputField(desc = "Indicates whether the 'E' label is accurate (True) or misapplied (False)")
    reformulated_query: str = dspy.OutputField(desc = "If the 'E' label is misapplied, provide a reformulated query that accurately matches the product; otherwise, null")

module = dspy.Predict(SearchAuditor)

In [18]:
# Testing the module on a single example
i = 8
query= df_data.iloc[i]['query']
product_title = df_data.iloc[i]['product_title']
product_description = df_data.iloc[i]['product_description']
product_bullet_points = df_data.iloc[i]['product_bullet_point']
esci_label = df_data.iloc[i]['esci_label']
product_brand = df_data.iloc[i]['product_brand']
product_color = df_data.iloc[i]['product_color']
tags = df_data.iloc[i]['tags']

response = module(
    query=query,
    product_title=product_title,
    product_description=product_description,
    product_bullet_points=product_bullet_points,
    esci_label=esci_label,
    product_brand=product_brand,
    product_color=product_color,
    tags=tags
)

print(query)
print(product_title)
print(product_description)
print(product_bullet_points)
print(esci_label)
print(product_brand)
print(product_color)
print(tags)
print(response.is_accurate)
print(response.reformulated_query)

dewalt 8v max cordless screwdriver kit, gyroscopic
DEWALT XTREME 12V MAX Cordless Screwdriver, 1/4-Inch, Tool Only (DCF601B)
nan
The cordless screwdriver has 25% more power**
The rechargeable screwdriver is 23% shorter**
Brushless motor of the powered screwdriver is designed for maximum runtime and durability
1/4-inch quick release drop and load hex that accepts 1-inch bit tips
15 clutch settings for a variety of fastening and drilling applications
E
DEWALT
nan
['12V MAX', 'Cordless', 'Screwdriver', '1/4-Inch', 'Brushless motor', 'Quick release', 'Clutch settings']
False
"dewalt 12v max cordless screwdriver kit"


In [19]:
# populating the data for all examples
results = []
for i in tqdm(range(len(df_data))):
    query= df_data.iloc[i]['query']
    product_title = df_data.iloc[i]['product_title']
    product_description = df_data.iloc[i]['product_description']
    product_bullet_points = df_data.iloc[i]['product_bullet_point']
    esci_label = df_data.iloc[i]['esci_label']
    product_brand = df_data.iloc[i]['product_brand']
    product_color = df_data.iloc[i]['product_color']
    tags = df_data.iloc[i]['tags']

    response = module(
        query=query,
        product_title=product_title,
        product_description=product_description,
        product_bullet_points=product_bullet_points,
        esci_label=esci_label,
        product_brand=product_brand,
        product_color=product_color,
        tags=tags
    )
    results.append({
        'query_id': df_data.iloc[i]['query_id'],
        'product_id': df_data.iloc[i]['product_id'],
        'is_accurate': response.is_accurate,
        'reformulated_query': response.reformulated_query,

        # extrat data for reference (can be commented out later)
        'query': query,
        'product_title': product_title,
        'product_description': product_description,
        'product_bullet_points': product_bullet_points,
        'esci_label': esci_label,
        'product_brand': product_brand,
        'product_color': product_color,
        'tags': tags
    })
df_results_single_llm_dspy = pd.DataFrame(results)

100%|██████████| 24/24 [00:00<00:00, 39.16it/s]


In [ ]:
df_results_single_llm_dspy.to_excel("./interim_results/shopping_queries_single_llm_dspy_results.xlsx", index=False)

### 3.2 Langchain Solution

In [21]:
from pydantic import BaseModel
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

In [22]:
class SearchOutput(BaseModel):
    is_accurate: bool
    reformulated_query: str


parser = JsonOutputParser(pydantic_object=SearchOutput)
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

llm = ChatOllama(model="gemma3:12b-it-qat", api_base="http://localhost:11434", api_key="")
chain = prompt | llm | parser

In [23]:
# Testing the module on a single example
i = 4
query= df_data.iloc[i]['query']
product_title = df_data.iloc[i]['product_title']
product_description = df_data.iloc[i]['product_description']
product_bullet_points = df_data.iloc[i]['product_bullet_point']
esci_label = df_data.iloc[i]['esci_label']
product_brand = df_data.iloc[i]['product_brand']
product_color = df_data.iloc[i]['product_color']
tags = df_data.iloc[i]['tags']

In [24]:
user_query = f"""
You are an e-commerce relevance auditor using the ESCI labeling framework. Check if the given query–product pair labeled “E” (Exact match) truly matches the definition:

Rules:
- “E” (Exact) = product title, description, and bullet points fully satisfy query intent. Extra or missing info is okay if not contradictory.
- NOT “E” = product contradicts query (e.g., includes excluded items, wrong specs, wrong dimensions/colors/variants).
- Ambiguous info = if product doesn’t contradict query, keep as “E”.
- Never hallucinate specs or infer unstated details.
- When “E” is incorrect, rewrite the query to accurately match the product title, product description and bullet points.
- add special attention to product specifications mentioned in the product title, description, bullet points, tags, product_brand, and prduct_color while reforming queries

Output format (JSON):
  "is_accurate": True/False,
  "reformulated_query": string or null

IMPORTANT:
Return ONLY a valid JSON dictionary No markdown.
No labels.
No explanation.
No extra text.
ONLY the JSON dictionary.
Output should not have leading or trailing token like ```json
Use the below examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.

Examples:

1. Inaccurate:
Query: 105" x 115" king spread without pillow shams
Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
Output:
"is_accurate": false, "reformulated_query": "105\" x 115\" king spread with pillow shams"

2. Accurate (extra info allowed):
Query: vitamina c
Product: Vitamina C Liposomal 400 mg, 90 capsules, extra details
Output:
"is_accurate": true, "reformulated_query": null

3. Accurate (missing info allowed):
Query: 12 month clothes girls
Product: Newborn kids romper + headband, no gender info
Output:
"is_accurate": true, "reformulated_query": null

4. Contradiction:
Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
Product: DEWALT 8V MAX, Gyroscopic
Output:
"is_accurate": false, "reformulated_query": "dewalt 8v max cordless screwdriver kit, gyroscopic"

Evaluate the following and produce only the JSON output:

query: {query}
product title: {product_title}
product description: {product_description}
product bullet points: {product_bullet_points}
ESCI label (always “E”): {esci_label}
product_brand: {product_brand}
product_color: {product_brand}
tags: {tags}
"""

In [25]:
output = chain.invoke({"query": user_query})

In [26]:
print(query)
print(product_title)
print(product_description)
print(product_bullet_points)
print(esci_label)
print(output['is_accurate'])
print(output['reformulated_query'])

aa batteries 100 pack
Rayovac AA Alkaline Double A Batteries, 60 Count
nan
60 pack of Rayovac High Energy Alkaline AA Batteries, Batteries AA Size
Long lasting batteries for high use devices and everyday electronics
Rayovac High Energy AA Batteries are ideal as flashlight batteries and in other high use devices, including wireless mice, remotes and toys
This AA battery pack holds power for up to 10 years
These double A batteries are designed to prevent damaging leaks and made in the USA with US and global parts
E
False
aa batteries 100 pack


In [27]:
# populating the data for all examples
results = []
for i in tqdm(range(len(df_data))):
    query= df_data.iloc[i]['query']
    product_title = df_data.iloc[i]['product_title']
    product_description = df_data.iloc[i]['product_description']
    product_bullet_points = df_data.iloc[i]['product_bullet_point']
    esci_label = df_data.iloc[i]['esci_label']
    product_brand = df_data.iloc[i]['product_brand']
    product_color = df_data.iloc[i]['product_color']
    tags = df_data.iloc[i]['tags']

    user_query = f"""
            You are an e-commerce relevance auditor using the ESCI labeling framework. Check if the given query–product pair labeled “E” (Exact match) truly matches the definition:

            Rules:
            - “E” (Exact) = product title, description, and bullet points fully satisfy query intent. Extra or missing info is okay if not contradictory.
            - NOT “E” = product contradicts query (e.g., includes excluded items, wrong specs, wrong dimensions/colors/variants).
            - Ambiguous info = if product doesn’t contradict query, keep as “E”.
            - Never hallucinate specs or infer unstated details.
            - When “E” is incorrect, rewrite the query to accurately match the product title, product description and bullet points.
            - add special attention to product specifications mentioned in the product title, description, bullet points, tags, product_brand, and prduct_color while reforming queries

            Output format (JSON):
            "is_accurate": True/False,
            "reformulated_query": string or null

            IMPORTANT:
            Return ONLY a valid JSON dictionary No markdown.
            No labels.
            No explanation.
            No extra text.
            ONLY the JSON dictionary.
            Output should not have leading or trailing token like ```json
            Use the below examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.

            Examples:

            1. Inaccurate:
            Query: 105" x 115" king spread without pillow shams
            Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
            Output:
            "is_accurate": false, "reformulated_query": "105\" x 115\" king spread with pillow shams"

            2. Accurate (extra info allowed):
            Query: vitamina c
            Product: Vitamina C Liposomal 400 mg, 90 capsules, extra details
            Output:
            "is_accurate": true, "reformulated_query": null

            3. Accurate (missing info allowed):
            Query: 12 month clothes girls
            Product: Newborn kids romper + headband, no gender info
            Output:
            "is_accurate": true, "reformulated_query": null

            4. Contradiction:
            Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
            Product: DEWALT 8V MAX, Gyroscopic
            Output:
            "is_accurate": false, "reformulated_query": "dewalt 8v max cordless screwdriver kit, gyroscopic"

            Evaluate the following and produce only the JSON output:

            query: {query}
            product title: {product_title}
            product description: {product_description}
            product bullet points: {product_bullet_points}
            ESCI label (always “E”): {esci_label}
            product_brand: {product_brand}
            product_color: {product_brand}
            tags: {tags}
    """

    response = chain.invoke({"query": user_query})

    try: 
        flag1 = 1 if response['is_accurate'] else 0
        flag2 = 1 if response['reformulated_query'] else 0
    except:
        print("Error in response parsing at index:", i)
        print("Response received:", response)
        response = chain.invoke({"query": user_query})

    results.append({
        'query_id': df_data.iloc[i]['query_id'],
        'product_id': df_data.iloc[i]['product_id'],
        'is_accurate': response['is_accurate'],
        'reformulated_query': response['reformulated_query'],

        # extrat data for reference (can be commented out later)
        'query': query,
        'product_title': product_title,
        'product_description': product_description,
        'product_bullet_points': product_bullet_points,
        'esci_label': esci_label,
        'product_brand': product_brand,
        'product_color': product_color,
        'tags': tags

    })
df_results_single_llm_langchain = pd.DataFrame(results)

 17%|█▋        | 4/24 [19:30<1:36:17, 288.86s/it]

Error in response parsing at index: 4
Response received: is_accurate


 42%|████▏     | 10/24 [53:57<1:19:12, 339.46s/it]

Error in response parsing at index: 10
Response received: is_accurate


 54%|█████▍    | 13/24 [1:10:34<57:27, 313.41s/it]  

Error in response parsing at index: 13
Response received: is_accurate


100%|██████████| 24/24 [2:10:09<00:00, 325.41s/it]  


In [28]:
df_results_single_llm_langchain.to_excel("./interim_results/shopping_queries_single_llm_langchain_results.xlsx", index=False)

### 3.3 Testing larger prompt

In [29]:
# Testing the module on a single example for larger prompt size
i = 4
query= df_data.iloc[i]['query']
product_title = df_data.iloc[i]['product_title']
product_description = df_data.iloc[i]['product_description']
product_bullet_points = df_data.iloc[i]['product_bullet_point']
esci_label = df_data.iloc[i]['esci_label']
product_brand = df_data.iloc[i]['product_brand']
product_color = df_data.iloc[i]['product_color']
tags = df_data.iloc[i]['tags']

In [30]:
# defining the tasks for LLM with appropriate prompts 
class SearchAuditor(dspy.Signature):
    """You are an expert e-commerce relevance auditor trained in the ESCI labeling framework. Your job is to verify whether query–product pairs labeled “E” (Exact match) truly satisfy the official definition:

        You will receive rows of data, each containing:
        1. query
        2. product title
        3. product description, 
        4. product bullet points
        5. ESCI label (always “E”)

        Your job:

        1. Evaluate whether the query is an exact match to the product according to the definition of label “E”. the label "E" stands for Exact match.
            An "Exact" match indicates that the product title, product description and product bullet points is a perfect or near-perfect match for the user's query and intent.
        2. Detect misapplied “E” labels — cases where a query specifies something contradictory or missing in the product.
        3. When inaccurate, rewrite the query so that it precisely matches the product as written.


        Remember the output format:
        json string containing 
          --is_accurate (True/False)
          --reformulated_query (only when False)
        You must generalize to unseen queries


        Here are some CRITICAL Gaurdrails you need to follow in you reasoning steps:

        What counts as “E” (Exact match)
        1. If the product fully satisfies all explicit requirements in the query.
        2. If the product has extra irrelevant info (e.g., includes optional bonus items).
        3. If the query mentions a specification that the product does NOT contradict, even if the product does not explicitly confirm it.
        4. If the product includes more items (e.g., comes with additional accessories) — still acceptable as “E”.

        What is NOT “E” (must detect mislabel)
        1. Product contains something explicitly contradictory to the query.
        2. Query requests a restriction the product violates.
        3. Query requests exclusion (“without X”) but product includes X.
        4. Query specifies pack sizes, dimensions, colors, or variants that conflict with product data.

        Ambiguity Rule

        1. If the product does not mention something the query specifies AND does not contradict it, leave it as “E”.


        You must NOT:
        1. invent or hallucinate product specs not present in data.
        2. infer unstated details (“probably includes batteries”) — only use explicit text.
        3. penalize missing product details unless they contradict the query.
        4. rewrite queries with guesses.
        5. deviate from the required output schema.


        Here are some example to help you understand the task better:

        Example 1 — Inaccurate Match
        Query: 105" x 115" king spread without pillow shams
        Product title: Greenland Home Moose Lodge Quilted Bedding Set, King, Natural
        Product Description: None
        Product bullet points: Each set includes quilt, plus two standard-sized 20x26-inch pillow shams\nComponent dimensions in inches (Quantity): 100x90-inch lightweight quilted coverlet + 20x26-inch standard sized pillow shams (2) [all +/-2 inches]\nImage shows a Full/Queen set on a full-sized bed. The fit and appearance of this product on your own bed may vary depending on bed size and product Size purchased.\nMachine quilted. Features cotton face, microfiber back and lightweight polyester fill 
        Product Brand: Greenland Home
        Product Color: Natural
        tags: ['King size', 'Quilted', 'Pillow shams included', 'Moose Lodge design', '105" x 115"', 'Bedding Set']
        Reason: contradicts query (“without pillow shams”)

        Correct Reformulated Query:
        105" x 115" king spread with pillow shams

        Output:
          "is_accurate": false,
          "reformulated_query": "105\" x 115\" king spread with pillow shams"


        Example 2 — Accurate Match (Extra info allowed in product metadata)
        Query: vitamina c
        Product title: Vitamina C Liposomal 400 mg | Dosis Elevada de Ácido L-Ascórbico | Vitamina C Altamente Biodisponible | Sistema Inmunológico | 90 cápsulas | Fabricado en Francia | Nutrivita
        Product Description: vitamina c 1000 mg liposomal pura vitamin 500mg niños vitaminas capsulas 1000mg liposomada nutrivita sistema inmunitario
        Product bullet points: ✅ ESTIMULE SUS DEFENSAS NATURALES. ¡La vitamina C es ideal para reforzar el tono y la vitalidad! Este potente antioxidante ayuda a reducir la fatiga, a mantener el buen funcionamiento del sistema inmunológico, a proteger las células contra el estrés oxidativo, a aumentar la absorción de hierro y a la formación normal de colágeno.\n💎 FÓRMULA ÚNICA. Nuestra vitamina C liposomal está formulada a partir de vitamina C (ácido L-ascórbico) encapsulada en liposomas de alta calidad a base der fosfolípidos de lecitina de girasol.\n🌿 VITAMINA C LIPOSOMAL ALTAMENTE ASIMILABLE. La encapsulación liposomal garantiza una absorción óptima de la vitamina C por el organismo. De hecho, la tasa de absorción de nuestra vitamina C liposomal alcanza el 90%. Y, al contrario de lo que ocurre don las formas "clásicas" de vitamina C, este suplemento no provoca molestias gástricas.\n🍊 ELEVADA DOSIS DE VITAMINA C. Cada cápsula te proporciona 400 mg de vitamina C liposomal. Esta elevada concentración permite aprovechar al máximo los beneficios de la vitamina C. Cada frasco contiene 90 cápsulas fáciles de tragar, un suministro completo para 3 meses.\n🇫🇷 DISEÑADO Y FABRICADO EN FRANCIA. Complemento alimenticio fabricado en un laboratorio francés según las normas más estrictas. A fin de obtener una calidad irreprochable, los suplementos de Nutrivita están garantizados como libres de OGM, sin gluten, sin alérgenos y sin estearato de magnesio. Aptos para dietas vegetarianas y veganas.
        Product Brand: Nutrivita
        Product Color: None
        tags: ['Vitamin C', 'Liposomal', '400 mg', '90 capsules', 'Immune System', 'Nutrivita']
        Reason: Extra items in product metadata do not violate the query

        Output:
          "is_accurate": true,
          "reformulated_query": null

        Example 3 — Accurate Despite Missing Info in Product metadata from query
        Query: 12 month clothes girls
        Product title: Newborn Kids Clothes Floral Jumpsuit Romper Playsuit + Headband Outfits (Blue Striped, 6-12 Months)
        Product Description: Attention plz: If your kid is chubby, we recomend choosing a larger size, thanks. Please kindly refer to your kids actual height and the size chart before buying/bidding. Thanks. Size Chart: Baby romper: Label Size 70 Tops Length 39 cm Bust 20 cm Recommended Age 0-6 Months Label Size 80 Tops Length 42 cm Bust 21cm Recommended Age 6-12 Months Label Size 90 Tops Length 43 cm Bust 22 cm Recommended Age 12-18 Months Label Size 100 Tops Length 45 cm Bust 23 cm Recommended Age 18-24 Months
        Product bullet points: Material: Polyester, Cotton. Ties closure. Soft baby clothes summer outfits, perfect baby birthday gift or baby shower gift, your little one will love this ruffle bodysuits for summer.\nStriped Romper.Big bowknot Style, Adjuastable back straps, Elastic waistline for comfortable fit,Make up your baby by this beautiful Romper, more attractive\nSuitable for babies about 0-2 years old. Your little one will get lots of compliments with this clothes set,affordable and durable.\nSize:70/80/90/100,different 4 size to choose, Lovely baby infant summer off-shoulder sleeve button romper jumpsuit short with headband cute summer clothes.\nPackage Included: 1 x Infant baby romper + 1 x cute headband.
        Product Brand: Lictin
        Product Color: Blue Striped
        tags: ['Newborn', 'Kids', 'Romper', 'Headband', '6-12 Months', 'cotton', 'Polyester']
        Reason: product does not contradict the query even tough it does not metion girls clothes specifically.

        Output:
          "is_accurate": true,
          "reformulated_query": null

        Example 4 — Contradiction in Product metadata from query
        Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
        Product title: DEWALT 8V MAX Cordless Screwdriver Kit, Gyroscopic, 1 Battery, Electric (DCF682N1)
        Product title: None
        Product bullet points:The cordless screwdriver features motion activation variable speed and reversing control for precise fastening control\nMotion activated variable speed 0-430 rpm of the rechargeable screwdriver is made for fastening into wood, plastic, and light-gauge metal\nThe powered screwdriver allows illumination in confined areas without shadowing\nBattery charge status on tool notifies when to charge packs\n1/4-inch hex allows for quick screwdriver bit change and holds 1-inch bit tips
        Product Brand: DEWALT
        Product Color: Yellow/Black
        tags: ['8V MAX', 'Cordless', 'Screwdriver kit', 'Gyroscopic', '1 Battery', 'Electric', 'DCF682N1']
        Reason: violates explicit “non-gyroscopic” requirement

        Correct Reformulated Query:
        dewalt 8v max cordless screwdriver kit, gyroscopic

        Output:
          "is_accurate": false,
          "reformulated_query": "dewalt 8v max cordless screwdriver kit, gyroscopic"

        Given the following data, evaluate the query–product pair, apply the rules above, detect incorrect “E” labels, and produce the required JSON output table. Do not return anything else.
      """

    query: str = dspy.InputField(desc = "User search query entered on platform")
    product_title: str=  dspy.InputField(desc = "Title of the product associated with the query")
    product_description: str = dspy.InputField(desc = "Description of the product associated with the query")
    product_bullet_points: str = dspy.InputField(desc = "Bullet points of the product associated with the query")
    esci_label: str= dspy.InputField(desc = "ESCI label assigned to the query-product pair, always 'E' in this case")
    product_brand: str= dspy.InputField(desc = "Brand of the product associated with the query")
    product_color: str= dspy.InputField(desc = "Color of the product associated with the query")
    tags: list[str]= dspy.InputField(desc = "List of product tags extracted from the product information")
    is_accurate: bool = dspy.OutputField(desc = "Indicates whether the 'E' label is accurate (True) or misapplied (False)")
    reformulated_query: str = dspy.OutputField(desc = "If the 'E' label is misapplied, provide a reformulated query that accurately matches the product; otherwise, null")

module = dspy.Predict(SearchAuditor)

In [31]:
# Testing the module on a single example
i = 8
query= df_data.iloc[i]['query']
product_title = df_data.iloc[i]['product_title']
product_description = df_data.iloc[i]['product_description']
product_bullet_points = df_data.iloc[i]['product_bullet_point']
esci_label = df_data.iloc[i]['esci_label']
product_brand = df_data.iloc[i]['product_brand']
product_color = df_data.iloc[i]['product_color']
tags = df_data.iloc[i]['tags']

response = module(
    query=query,
    product_title=product_title,
    product_description=product_description,
    product_bullet_points=product_bullet_points,
    esci_label=esci_label,
    product_brand=product_brand,
    product_color=product_color,
    tags=tags
)

print(query)
print(product_title)
print(product_description)
print(product_bullet_points)
print(esci_label)
print(product_brand)
print(product_color)
print(tags)
print(response.is_accurate)
print(response.reformulated_query)

dewalt 8v max cordless screwdriver kit, gyroscopic
DEWALT XTREME 12V MAX Cordless Screwdriver, 1/4-Inch, Tool Only (DCF601B)
nan
The cordless screwdriver has 25% more power**
The rechargeable screwdriver is 23% shorter**
Brushless motor of the powered screwdriver is designed for maximum runtime and durability
1/4-inch quick release drop and load hex that accepts 1-inch bit tips
15 clutch settings for a variety of fastening and drilling applications
E
DEWALT
nan
['12V MAX', 'Cordless', 'Screwdriver', '1/4-Inch', 'Brushless motor', 'Quick release', 'Clutch settings']
False
dewalt 8v max cordless screwdriver kit, non-gyroscopic


## 4. Conclusions

#### 1. The following models were tested on different prompts. The LLMs were choosen based on https://artificialanalysis.ai/leaderboards/models?open_weights=open_source&size_class=tiny benchmarks
    1.1 Gemma3:12b-it-qat
    1.2 Gemma3:4b
    1.3 Gemma3:1b
    1.4 Phi4-mini: 4b
    1.5 Qwen3 4b
#### 2. The Qwen3 4b was a reasoning model but I did not chose it because reasoning model takes time to get an output due to it's thinking capabilities specially when I was limited by only running LLMs locally on 16GB Macbook Pro. 
#### 3. The follwing promting strategies were tested to come to final prompt
    3.1 Zero shot prompting with tasks defined.
    3.2 Addition of constrains to add Gaurdrails. 
    3.3 Adding output specifications make output parsable and short. 
    3.4 Self Critique and internal multi step reasoning. 
    3.5 Adding examples to mimic Few shot prompting.
#### 4. Observations and Decision making: 
    4.1 A smaller prompt was not able to provide proper instructions for LLM to complete task fully. 
    4.2 1B models were not able to capture instuctions without furhter finetuning (LoRa). It was kept out of scope for this assesment for now but smaller LLMs can perform better if we have lablled data and finetune them. 
    4.3 4B models were able to perform better when it came to capturing smaller prompts with simppler tasks. But as the prompts grew they failed when I incorporated few shot examples. 
    4.3 12B model was not able to run on my macbook and was very slow. To increase the response time, I used the quantized version of the model to decrease memory footprint without affecting accuracy much. 
#### 5. While choosing few show example, I ensured that we don't take direct examples from our data set as that would not lead to genralized model. In fact I choosed examples from remaining data to ensure genralized model. Using that I tested on given data. 
#### 6. I choose DsPy and langchain frameworks to complete the task. In my opinion both framworks are great for developing the AI applications. Usually in a corporate scenario we test out multiple framworks that fit with given infrastructure and with this exersise, I wanted to show that.
    6.1 While langchain has better open source community support, In my personal opinion I found DsPy to be equivalent. Also DsPy is better for a for someone who is a Data scientist and has experience working with Pytorch since it frames the code in programing manner rather than simple prompting.

#### 7. I also tested a a much larger prompt to test if adding more information to 12B LLM may be helpful for the purpose of this task. I found the the medium sized prompt worked similar for the given data as larger prompt. Thus to reduce the input token usege I choose to stick with a medium sised prompt. 



# 5. Building Search Auditor using multiple LLMs (More Testing on smaller LLMs)

In [32]:
lm = dspy.LM("ollama_chat/gemma3:4b", api_base="http://localhost:11434", api_key="") # both of the models used in my case are hosted on 11434 port but if you server on different port please change accordingly
dspy.configure(lm=lm, temperature=0)

In [33]:
class SearchAuditorIsAccusrate(dspy.Signature):
    """You are an e-commerce relevance auditor using the ESCI labeling framework. Check if the given query–product pair labeled “E” (Exact match) truly matches the definition:

        Rules:
        - “E” (Exact) = product title, description, and bullet points fully satisfy query intent. Extra or missing info is okay if not contradictory.
        - NOT “E” = product contradicts query (e.g., includes excluded items, wrong specs, wrong dimensions/colors/variants).
        - Ambiguous info = if product doesn’t contradict query, keep as “E”.
        - Never hallucinate specs or infer unstated details.

        Output format (JSON):
          "is_accurate": True/False

        Examples:

        1. Inaccurate:
        Query: 105" x 115" king spread without pillow shams
        Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
        Output:
        "is_accurate": false

        2. Accurate (extra info allowed):
        Query: vitamina c
        Product: Vitamina C Liposomal 400 mg, 90 capsules, extra details
        Output:
        "is_accurate": true

        3. Accurate (missing info allowed):
        Query: 12 month clothes girls
        Product: Newborn kids romper + headband, no gender info
        Output:
        "is_accurate": true

        4. Contradiction:
        Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
        Product: DEWALT 8V MAX, Gyroscopic
        Output:
        "is_accurate": false

      Use the above examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.
      """

    query: str = dspy.InputField(desc = "User search query entered on platform")
    product_title: str=  dspy.InputField(desc = "Title of the product associated with the query")
    product_description: str = dspy.InputField(desc = "Description of the product associated with the query")
    product_bullet_points: str = dspy.InputField(desc = "Bullet points of the product associated with the query")
    esci_label: str= dspy.InputField(desc = "ESCI label assigned to the query-product pair, always 'E' in this case")
    product_brand: str= dspy.InputField(desc = "Brand of the product associated with the query")
    product_color: str= dspy.InputField(desc = "Color of the product associated with the query")
    tags: list[str]= dspy.InputField(desc = "List of product tags extracted from the product information")
    is_accurate: bool = dspy.OutputField(desc = "Indicates whether the 'E' label is accurate (True) or misapplied (False)")

module_accurate = dspy.Predict(SearchAuditorIsAccusrate)

In [34]:
class SearchAuditorReformulateQuery(dspy.Signature):
    """You are an e-commerce query generator. Check if the given query–product pair generate a reformulated query that a user may put in to find the given product.:

        Rules:
        - keep the output concise and relevant to the product details.
        - add special attention to product specifications mentioned in the product title, description, and bullet points while reforming queries
        - avoid unnecessary elaboration or verbosity.
        - ensure the reformulated query accurately reflects the product's key features and specifications.
        - Never hallucinate specs or infer unstated details.
        - add special attention to product specifications mentioned in the product title, description, bullet points, tags, product_brand, and prduct_color while reforming queries

        Output format (JSON):
          "reformulated_query": string or null

        Examples:

        1. Inaccurate:
        Query: 105" x 115" king spread without pillow shams
        Product: Greenland Home Moose Lodge Quilted Bedding Set, King, includes pillow shams
        Output:
        "reformulated_query": "105\" x 115\" king spread with pillow shams"

        2. Contradiction:
        Query: dewalt 8v max cordless screwdriver kit, non-gyroscopic
        Product: DEWALT 8V MAX, Gyroscopic
        Output:
        "reformulated_query": "dewalt 8v max cordless screwdriver kit, gyroscopic"

      Use the above examples as reference to guide your evaluations. DO NOT directly think that they are the answers rather understand the reasoning behind the output decisions.
      """

    query: str = dspy.InputField(desc = "User search query entered on platform.")
    product_title: str=  dspy.InputField(desc = "Title of the product associated with the query")
    product_description: str = dspy.InputField(desc = "Description of the product associated with the query")
    product_bullet_points: str = dspy.InputField(desc = "Bullet points of the product associated with the query")
    esci_label: str= dspy.InputField(desc = "ESCI label assigned to the query-product pair, always 'E' in this case")
    product_brand: str= dspy.InputField(desc = "Brand of the product associated with the query")
    product_color: str= dspy.InputField(desc = "Color of the product associated with the query")
    tags: list[str]= dspy.InputField(desc = "List of product tags extracted from the product information")
    reformulated_query: str = dspy.OutputField(desc = "provide a reformulated query that accurately matches the product; otherwise, null")

module_reformulation = dspy.Predict(SearchAuditorReformulateQuery)

In [35]:
# Testing the module on a single example for larger prompt size
i = 4
query= df_data.iloc[i]['query']
product_title = df_data.iloc[i]['product_title']
product_description = df_data.iloc[i]['product_description']
product_bullet_points = df_data.iloc[i]['product_bullet_point']
esci_label = df_data.iloc[i]['esci_label']
product_brand = df_data.iloc[i]['product_brand']
product_color = df_data.iloc[i]['product_color']
tags = df_data.iloc[i]['tags']

In [36]:
response_accurate = module_accurate(
    query=query,
    product_title=product_title,
    product_description=product_description,
    product_bullet_points=product_bullet_points,
    esci_label=esci_label,
    product_brand=product_brand,
    product_color=product_color,
    tags=tags
)

print(query)
print(product_title)
print(product_description)
print(product_bullet_points)
print(esci_label)
print(product_brand)
print(product_color)
print(tags)
print(response_accurate.is_accurate)

aa batteries 100 pack
Rayovac AA Alkaline Double A Batteries, 60 Count
nan
60 pack of Rayovac High Energy Alkaline AA Batteries, Batteries AA Size
Long lasting batteries for high use devices and everyday electronics
Rayovac High Energy AA Batteries are ideal as flashlight batteries and in other high use devices, including wireless mice, remotes and toys
This AA battery pack holds power for up to 10 years
These double A batteries are designed to prevent damaging leaks and made in the USA with US and global parts
E
Rayovac
nan
['AA batteries', 'Alkaline', '60 count', 'Rayovac', 'Long lasting', 'Flashlight', 'Wireless', 'Remotes', 'Toys', '10 years', 'Leak proof', 'USA made']
False


In [37]:
response_reformulation = module_reformulation(
    query=query,
    product_title=product_title,
    product_description=product_description,
    product_bullet_points=product_bullet_points,
    esci_label=esci_label,
    product_brand=product_brand,
    product_color=product_color,
    tags=tags
)

print(query)
print(product_title)
print(product_description)
print(product_bullet_points)
print(esci_label)
print(product_brand)
print(product_color)
print(tags)
print(response_reformulation.reformulated_query)

aa batteries 100 pack
Rayovac AA Alkaline Double A Batteries, 60 Count
nan
60 pack of Rayovac High Energy Alkaline AA Batteries, Batteries AA Size
Long lasting batteries for high use devices and everyday electronics
Rayovac High Energy AA Batteries are ideal as flashlight batteries and in other high use devices, including wireless mice, remotes and toys
This AA battery pack holds power for up to 10 years
These double A batteries are designed to prevent damaging leaks and made in the USA with US and global parts
E
Rayovac
nan
['AA batteries', 'Alkaline', '60 count', 'Rayovac', 'Long lasting', 'Flashlight', 'Wireless', 'Remotes', 'Toys', '10 years', 'Leak proof', 'USA made']
Rayovac 60 pack of AA Alkaline Batteries


In [38]:
# populating the data for all examples
results = []
for i in tqdm(range(len(df_data))):
    query= df_data.iloc[i]['query']
    product_title = df_data.iloc[i]['product_title']
    product_description = df_data.iloc[i]['product_description']
    product_bullet_points = df_data.iloc[i]['product_bullet_point']
    esci_label = df_data.iloc[i]['esci_label']
    product_brand = df_data.iloc[i]['product_brand']
    product_color = df_data.iloc[i]['product_color']
    tags = df_data.iloc[i]['tags']

    response_accurate = module_accurate(
        query=query,
        product_title=product_title,
        product_description=product_description,
        product_bullet_points=product_bullet_points,
        esci_label=esci_label,
        product_brand=product_brand,
        product_color=product_color,
        tags=tags
    )

    if not response_accurate.is_accurate:
        response_reformulation = module_reformulation(
            query=query,
            product_title=product_title,
            product_description=product_description,
            product_bullet_points=product_bullet_points,
            esci_label=esci_label,
            product_brand=product_brand,
            product_color=product_color,
            tags=tags
        )

    results.append({
        'query_id': df_data.iloc[i]['query_id'],
        'product_id': df_data.iloc[i]['product_id'],
        'is_accurate': response_accurate.is_accurate,
        'reformulated_query': response_reformulation.reformulated_query if not response_accurate.is_accurate else None,

        # extrat data for reference (can be commented out later)
        'query': query,
        'product_title': product_title,
        'product_description': product_description,
        'product_bullet_points': product_bullet_points,
        'esci_label': esci_label,
        'product_brand': product_brand,
        'product_color': product_color,
        'tags': tags
    })
df_results_multiple_llm_dspy = pd.DataFrame(results)

100%|██████████| 24/24 [00:01<00:00, 18.18it/s]


In [ ]:
df_results_multiple_llm_dspy.to_excel("./interim_results/shopping_queries_multiple_llm_dspy_results.xlsx", index=False)

## 5.1 Conclusion Multiple LLM

1. Looking at the results, We can see that dividing the task to saperate 2 prompts helped in getting better results compared to combined prompt with 4B model.
2. On comparision of results to 12B model, 12B models results are still better at generating the reformulated query. 
3. Since we we used 4B model with multiple prompt strategy it was much faster compared to 12B model results. 
4. The best strategy would be to use multiple prompts by splitting the tasks with 12B model but that will take 2x time with 12B model. So I have put that out of scope for now. 

In [ ]:
# final output saving
df_results_single_llm_dspy[[
    'query_id', 'product_id', 'is_accurate', 'reformulated_query'
]].to_excel("./output_data/shopping_queries_result.xlsx", index=False)
# 7, 16